### TRANSFERIR LOS DATOS A LA CAPA BRONZE
**IMPORTAMOS LAS LIBRERIAS**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import time
from datetime import datetime

**EXTRAEMOS LOS DATOS DE LA CAPA RAW**

In [0]:
current_date = datetime.utcnow().date()
raw_dir = "spotify_raw_"+str(current_date.year) + "_" + str(current_date.month) + "_" + str(current_date.day) 

dbutils.widgets.text("raw_table", raw_dir, "Raw Table")

In [0]:
raw = "proyecto_spotify.raw." + dbutils.widgets.get("raw_table")
bronze_raw = spark.table(raw)

display(bronze_raw.limit(5))

**APLANAMIENTO DE ESTRUCTURAS JSON (FLATTENING)**

In [0]:
bronze_tracks = (
    bronze_raw
    .select(
        "search_term",
        "offset",
        "extraction_timestamp",
        from_json(
            "payload",
            """
            STRUCT<
                tracks: STRUCT<
                    items: ARRAY<STRUCT<
                        id: STRING,
                        name: STRING,
                        popularity: INT,
                        duration_ms: BIGINT,
                        explicit: BOOLEAN,
                        uri: STRING,
                        href: STRING,
                        external_urls: STRUCT<
                            spotify: STRING
                        >,
                        album: STRUCT<
                            id: STRING,
                            name: STRING,
                            album_type: STRING,
                            release_date: STRING,
                            release_date_precision: STRING,
                            total_tracks: INT,
                            href: STRING,
                            uri: STRING
                        >,
                        artists: ARRAY<STRUCT<
                            id: STRING,
                            name: STRING,
                            href: STRING,
                            uri: STRING
                        >>
                    >>
                >
            >
            """
        ).alias("data")
    )
)

#display(bronze_tracks.limit(5))

In [0]:
bronze_tracks = (
    bronze_tracks
    .select(
        "search_term",
        "extraction_timestamp",
        explode("data.tracks.items").alias("track")
    )
)

#display(bronze_tracks.limit(5))

In [0]:
bronze_tracks = (
    bronze_tracks
    .select(
        "search_term",
        "extraction_timestamp",
        "track",
        explode("track.artists").alias("artist")
    )
)

#display(bronze_tracks.limit(5))

In [0]:
bronze_tracks_final = bronze_tracks.select(
    # Datos del track
    col("track.id").alias("track_id"),
    col("track.name").alias("track_name"),
    col("track.popularity").alias("popularity"),
    col("track.duration_ms").alias("duration_ms"),
    col("track.explicit").alias("explicit"),
    col("track.uri").alias("track_uri"),
    col("track.external_urls.spotify").alias("spotify_url"),

    # Datos del album
    col("track.album.id").alias("album_id"),
    col("track.album.name").alias("album_name"),
    col("track.album.album_type").alias("album_type"),
    col("track.album.release_date").alias("release_date"),
    col("track.album.total_tracks").alias("album_total_tracks"),

    # Datos del artist
    col("artist.id").alias("artist_id"),
    col("artist.name").alias("artist_name"),
    col("artist.href").alias("artist_href"),
    col("artist.uri").alias("artist_uri"),

    # Columnas de Auditoria
    "search_term",
    "extraction_timestamp"
)

#display(bronze_tracks_final.limit(10))

In [0]:
print("Cantidad de Registros en Capa Raw: ", bronze_raw.count())
print("Cantidad de Registros en Capa Bronze: ", bronze_tracks_final.count())

**GUARDAMOS LOS DATOS EN LA CAPA BRONZE**

In [0]:
(
    bronze_tracks_final
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(
        "proyecto_spotify.bronze.spotify_tracks"
    )
)